In [1]:
import requests
import pandas as pd
import io
import time
import matplotlib.pyplot as plt

In [11]:
from google.colab import auth
auth.authenticate_user()


In [2]:
#Get API from USGS
url = "https://earthquake.usgs.gov/fdsnws/event/1/query"

start_year = 2020
end_year = 2025  # Adjust to the latest date
min_magnitude = 4.0  # Adjust as needed

# Define batch sizes based on data volume
batch_sizes = {
    (2020, 2025): 1,
}

#store dataframes
all_data = []

# Loop through different batch sizes and store in all_data
for (start, end), batch_size in batch_sizes.items():
    for year in range(start, end + 1, batch_size):
        start_time = f"{year}-01-01"
        end_time = f"{min(year + batch_size - 1, end_year)}-12-31"

        params = {
            "format": "csv",
            "starttime": start_time,
            "endtime": end_time,
            "minmagnitude": min_magnitude,
            "maxmagnitude": 10.0,
            "orderby": "time"
        }

        print(f"Fetching data from {start_time} to {end_time}...")

        response = requests.get(url, params=params)

        if response.status_code == 200:
            # Convert response content to dataframe
            df = pd.read_csv(io.StringIO(response.text))
            all_data.append(df)

            print(f"✅ Data fetched for {start_time} to {end_time} ({len(df)} records)")
        else:
            print(f"❌ Failed for {start_time} to {end_time}: {response.status_code}")

        # Sleep to avoid hitting API rate limits
        time.sleep(2)

# Merge all data
if all_data:
    final_df = pd.concat(all_data, ignore_index=True)
    final_df.to_csv("earthquake_data_global.csv", index=False)
    print("✅ All data saved: earthquake_data_global.csv")
else:
    print("❌ No data collected.")




Fetching data from 2020-01-01 to 2020-12-31...
✅ Data fetched for 2020-01-01 to 2020-12-31 (13918 records)
Fetching data from 2021-01-01 to 2021-12-31...
✅ Data fetched for 2021-01-01 to 2021-12-31 (17244 records)
Fetching data from 2022-01-01 to 2022-12-31...
✅ Data fetched for 2022-01-01 to 2022-12-31 (15715 records)
Fetching data from 2023-01-01 to 2023-12-31...
✅ Data fetched for 2023-01-01 to 2023-12-31 (16191 records)
Fetching data from 2024-01-01 to 2024-12-31...
✅ Data fetched for 2024-01-01 to 2024-12-31 (14132 records)
Fetching data from 2025-01-01 to 2025-12-31...
✅ Data fetched for 2025-01-01 to 2025-12-31 (3631 records)
✅ All data saved: earthquake_data_global.csv


In [3]:
# Load the full earthquake dataset
df_eq = pd.read_csv("earthquake_data_global.csv")

# Define Myanmar's geographic boundaries (approximate lat/lon range)
myanmar_bounds = (16.0, 28.0, 92.0, 101.0)

# Filter for earthquakes within Myanmar
df_myanmar = df_eq[
    (df_eq["latitude"] >= myanmar_bounds[0]) & (df_eq["latitude"] <= myanmar_bounds[1]) &
    (df_eq["longitude"] >= myanmar_bounds[2]) & (df_eq["longitude"] <= myanmar_bounds[3])
]

# Save the Myanmar-specific earthquake dataset
df_myanmar.to_csv("earthquake_data_myanmar.csv", index=False)

print(f"✅ Filtered dataset saved: earthquake_data_myanmar.csv ({len(df_myanmar)} records)")
df_myanmar.head()


✅ Filtered dataset saved: earthquake_data_myanmar.csv (557 records)


,time,latitude,longitude,depth,mag,magType,nst,gap,dmin,rms,...,updated,place,type,horizontalError,depthError,magError,magNst,status,locationSource,magSource
172,2020-12-26T21:48:15.074Z,27.3034,96.9396,10.0,4.5,mb,NaN,113.0,3.876,0.66,...,2021-03-06T23:19:18.040Z,"93 km SSE of Hayuliang, India",earthquake,10.2,1.9,0.139,15.0,reviewed,us,us
181,2020-12-26T14:19:09.417Z,16.0246,92.9758,10.0,4.5,mb,NaN,64.0,4.348,1.24,...,2021-03-06T23:19:10.040Z,"205 km WSW of Pathein, Myanmar",earthquake,4.7,1.8,0.163,11.0,reviewed,us,us
271,2020-12-24T15:13:50.935Z,25.4629,96.6657,10.0,4.4,mb,NaN,46.0,2.475,1.08,...,2021-03-06T23:18:20.040Z,"74 km W of Myitkyina, Myanmar",earthquake,3.3,1.9,0.079,58.0,reviewed,us,us
274,2020-12-24T12:47:17.072Z,25.6158,96.6570,10.0,4.3,mb,NaN,121.0,2.546,1.34,...,2021-03-06T23:18:19.040Z,"67 km SSW of Sarupathar, India",earthquake,5.1,1.9,0.133,16.0,reviewed,us,us
279,2020-12-24T08:09:48.075Z,25.6397,96.6534,10.0,4.8,mb,NaN,45.0,2.556,0.74,...,2021-03-06T23:18:15.040Z,"64 km SSW of Sarupathar, India",earthquake,7.2,1.9,0.066,74.0,reviewed,us,us


In [4]:
import folium
import pandas as pd
import ipywidgets as widgets
from IPython.display import display

# Load the Myanmar earthquake dataset
df_myanmar = pd.read_csv("earthquake_data_myanmar.csv")

# Convert 'time' column to datetime format (remove timezone issues)
df_myanmar["time"] = pd.to_datetime(df_myanmar["time"]).dt.tz_localize(None)

# Get min and max date from the dataset
min_date = df_myanmar["time"].min().date()
max_date = df_myanmar["time"].max().date()

# Create a date range slider
date_slider = widgets.SelectionRangeSlider(
    options=[d.strftime('%Y-%m-%d') for d in pd.date_range(min_date, max_date)],
    index=(0, len(pd.date_range(min_date, max_date)) - 1),
    description="Date Range",
    layout={'width': '600px'}
)

# Function to update the map based on selected date range
def update_map(date_range):
    start, end = map(pd.Timestamp, date_range)  # Convert to timestamps
    filtered_df = df_myanmar[(df_myanmar["time"] >= start) & (df_myanmar["time"] <= end)]

    # Create a base map centered on Myanmar
    myanmar_map = folium.Map(location=[21.0, 96.0], zoom_start=6)

    # Add earthquake markers
    for _, row in filtered_df.iterrows():
        folium.CircleMarker(
            location=[row["latitude"], row["longitude"]],
            radius=row["mag"] * 1.5,
            color="red",
            fill=True,
            fill_color="red",
            fill_opacity=0.6,
            popup=f"Date: {row['time'].date()}<br>Magnitude: {row['mag']}<br>Depth: {row['depth']} km<br>Location: {row['place']}",
        ).add_to(myanmar_map)

    display(myanmar_map)

# Interactive widget
widgets.interactive(update_map, date_range=date_slider)


interactive(children=(SelectionRangeSlider(description='Date Range', index=(0, 1902), layout=Layout(width='600…

Load population data and calculate affected population

In [6]:
import pandas as pd
from geopy.distance import geodesic # Import geodesic

# Load the dataframe from your Google Drive
df_pop_mm = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/myamarpop_2019Intercensus.csv')

# Drop columns without names (usually represented by Unnamed: #)
df_pop_mm = df_pop_mm.loc[:, ~df_pop_mm.columns.str.startswith('Unnamed')]

# Now you can work with the df_pop DataFrame
df_pop_mm.head()

# Filter population data for specific regions: Sagaing, Mandalay, Bago
regions_of_interest = ["Sagaing", "Mandalay", "Bago"]
df_pop_filtered = df_pop_mm[df_pop_mm["State/Region"].isin(regions_of_interest)]

# Function to estimate affected population based on distance
def estimate_affected_population(eq_lat, eq_lon, eq_mag):
    affected_pop = 0

    for _, row in df_pop_filtered.iterrows():
        region_lat = row["Latitude"]
        region_lon = row["Longitude"]
        # Access the population data from the "Both sexes" column
        total_pop = row["Both sexes"]

        # Calculate distance (km) from earthquake to the region center
        distance_km = geodesic((eq_lat, eq_lon), (region_lat, region_lon)).km

        # Define an impact weight based on distance
        if distance_km <= 50:
            weight = 1  # High impact (full population affected)
        elif distance_km <= 100:
            weight = 0.5  # Medium impact
        elif distance_km <= 200:
            weight = 0.2  # Low impact
        else:
            weight = 0  # No significant impact

        # Estimate affected population
        affected_pop += total_pop * weight * (eq_mag / 7.0)  # Scale by magnitude

    return int(affected_pop)

# Apply function to all earthquakes
df_eq["Estimated_Affected_Pop"] = df_eq.apply(
    lambda row: estimate_affected_population(row["latitude"], row["longitude"], row["mag"]), axis=1
)

# Save results
df_eq.to_csv("earthquake_affected_population_filtered.csv", index=False)
print("✅ Estimated affected population saved in 'earthquake_affected_population_filtered.csv'")

# Display results
df_eq.tail()


✅ Estimated affected population saved in 'earthquake_affected_population_filtered.csv'


,time,latitude,longitude,depth,mag,magType,nst,gap,dmin,rms,...,place,type,horizontalError,depthError,magError,magNst,status,locationSource,magSource,Estimated_Affected_Pop
80826,2025-01-01T02:00:06.918Z,-3.8521,151.6207,10.000,4.5,mb,36.0,98.0,0.591,0.98,...,"71 km WNW of Rabaul, Papua New Guinea",earthquake,7.80,1.901,0.116,22.0,reviewed,us,us,0
80827,2025-01-01T01:46:59.538Z,-3.8682,151.6536,10.000,4.7,mb,39.0,99.0,0.557,0.86,...,"67 km WNW of Rabaul, Papua New Guinea",earthquake,6.86,1.901,0.108,26.0,reviewed,us,us,0
80828,2025-01-01T01:07:10.492Z,-5.9077,128.8869,277.355,4.4,mb,19.0,81.0,1.709,0.89,...,"256 km SSE of Ambon, Indonesia",earthquake,10.20,12.427,0.203,7.0,reviewed,us,us,0
80829,2025-01-01T01:00:50.839Z,34.6287,16.5823,10.000,4.0,mb,12.0,246.0,3.361,1.19,...,"228 km SE of Marsaskala, Malta",earthquake,7.78,2.005,0.211,6.0,reviewed,us,us,0
80830,2025-01-01T00:13:22.650Z,12.7441,143.6244,35.000,4.2,mb,12.0,160.0,1.474,0.76,...,"127 km WSW of Merizo Village, Guam",earthquake,12.60,1.974,0.199,7.0,reviewed,us,us,0


Weighted Affected People for All Years (2020 - 2025)

In [7]:
import pandas as pd
import numpy as np

# Ensure earthquake time is in datetime format
df_eq["time"] = pd.to_datetime(df_eq["time"], errors="coerce")

def estimate_affected_population(eq_lat, eq_lon, eq_mag):
    affected_pop = 0

    for _, row in df_pop_mm.iterrows():
        region_lat = row["Latitude"]
        region_lon = row["Longitude"]
        total_pop = row["Both sexes"]

        # Calculate distance from earthquake to region center
        distance_km = geodesic((eq_lat, eq_lon), (region_lat, region_lon)).km

        # Define dynamic buffer based on earthquake magnitude
        max_radius = eq_mag * 10  # Example: M6.5 -> 65 km radius

        if distance_km <= max_radius:
            # Use an exponential decay function to reduce impact with distance
            weight = np.exp(-distance_km / (max_radius / 2))

            # Scale by magnitude factor
            magnitude_factor = (eq_mag / 7.0) ** 2

            # Compute estimated affected population
            affected_pop += total_pop * weight * magnitude_factor

    return int(affected_pop)

# 🔹 **Apply function to all earthquakes (all years)**
df_eq["Estimated_Affected_Pop"] = df_eq.apply(
    lambda row: estimate_affected_population(row["latitude"], row["longitude"], row["mag"]), axis=1
)

# 🔹 **Save results**
df_eq.to_csv("earthquake_affected_population_weighted_all_years.csv", index=False)
print("✅ Estimated affected population for all years saved in 'earthquake_affected_population_weighted_all_years.csv'")

# 🔹 **Display results**
df_eq[["time", "latitude", "longitude", "mag", "Estimated_Affected_Pop"]].tail()


✅ Estimated affected population for all years saved in 'earthquake_affected_population_weighted_all_years.csv'


,time,latitude,longitude,mag,Estimated_Affected_Pop
80826,2025-01-01 02:00:06.918000+00:00,-3.8521,151.6207,4.5,0
80827,2025-01-01 01:46:59.538000+00:00,-3.8682,151.6536,4.7,0
80828,2025-01-01 01:07:10.492000+00:00,-5.9077,128.8869,4.4,0
80829,2025-01-01 01:00:50.839000+00:00,34.6287,16.5823,4.0,0
80830,2025-01-01 00:13:22.650000+00:00,12.7441,143.6244,4.2,0


In [10]:
import folium
import pandas as pd
import numpy as np
import ipywidgets as widgets
from folium.plugins import HeatMap
from geopy.distance import geodesic
from IPython.display import display
from datetime import datetime

# Load earthquake data
df_eq = pd.read_csv("earthquake_affected_population_weighted_all_years.csv")

# Convert 'time' column to datetime and remove timezone
df_eq["time"] = pd.to_datetime(df_eq["time"], errors="coerce")
df_eq["time"] = df_eq["time"].dt.tz_localize(None)

# Filter for 2024 and 2025 earthquakes
df_eq_filtered = df_eq[df_eq["time"].dt.year.isin([2024, 2025])].copy()

# Load population dataset
df_pop_mm = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/myamarpop_2019Intercensus.csv')

# 🔹 **Population Estimation Function (No Region Filter)**
def estimate_affected_population(eq_lat, eq_lon, eq_mag):
    affected_pop = 0
    selected_pop_data = df_pop_mm.copy()  # Use entire dataset

    # Ensure numeric types
    selected_pop_data["Latitude"] = pd.to_numeric(selected_pop_data["Latitude"], errors="coerce")
    selected_pop_data["Longitude"] = pd.to_numeric(selected_pop_data["Longitude"], errors="coerce")
    selected_pop_data["Both sexes"] = pd.to_numeric(selected_pop_data["Both sexes"], errors="coerce")

    print(f"\n🔍 Earthquake at ({eq_lat}, {eq_lon}) M{eq_mag}")
    print(f"📌 Analyzing {len(selected_pop_data)} population points...")

    for _, row in selected_pop_data.iterrows():
        region_lat = row["Latitude"]
        region_lon = row["Longitude"]
        total_pop = row["Both sexes"]

        # Skip invalid data
        if pd.isna(region_lat) or pd.isna(region_lon) or pd.isna(total_pop):
            continue

        # Calculate distance from earthquake to region center
        distance_km = geodesic((eq_lat, eq_lon), (region_lat, region_lon)).km

        # Define dynamic buffer based on earthquake magnitude
        max_radius = eq_mag * 10  # Example: M6.5 -> 65 km radius

        if distance_km <= max_radius:
            # Use an exponential decay function to reduce impact with distance
            weight = np.exp(-distance_km / (max_radius / 2))
            magnitude_factor = (eq_mag / 7.0) ** 2  # Scale by magnitude

            affected_pop += total_pop * weight * magnitude_factor

            print(f"🌍 Region at ({region_lat}, {region_lon}) | Pop: {total_pop}")
            print(f"   ➡️ Distance: {distance_km:.2f} km, Weight: {weight:.4f}, Mag Factor: {magnitude_factor:.4f}")

    print(f"🎯 Estimated Affected Population: {int(affected_pop)}\n")
    return int(affected_pop)


# 🔹 **Map Visualization Function**
def plot_earthquake_map(start_date=None, end_date=None):
    # Default dates: 2025-01-01 to today
    if start_date is None:
        start_date = datetime(2025, 1, 1)
    if end_date is None:
        end_date = datetime.today()

    start_date = pd.to_datetime(start_date)
    end_date = pd.to_datetime(end_date)

    # Filter earthquakes by selected date range
    filtered_eq = df_eq_filtered[(df_eq_filtered["time"] >= start_date) & (df_eq_filtered["time"] <= end_date)].copy()

    if filtered_eq.empty:
        print("⚠️ No earthquakes found in the selected time range.")
        return None

    # Compute affected population for each earthquake (without region filtering)
    filtered_eq["Weighted_Affected_Pop"] = filtered_eq.apply(
        lambda row: estimate_affected_population(row["latitude"], row["longitude"], row["mag"]),
        axis=1
    )

    # Ensure affected population is not all zeros
    print("📊 Min Affected Pop:", filtered_eq["Weighted_Affected_Pop"].min())
    print("📊 Max Affected Pop:", filtered_eq["Weighted_Affected_Pop"].max())

    # Create Folium Map
    myanmar_map = folium.Map(location=[21.0, 96.0], zoom_start=6)

    # 🔹 **Add Population Heatmap**
    heat_data = [
        [row["Latitude"], row["Longitude"], row["Both sexes"]]
        for _, row in df_pop_mm.iterrows()
    ]

    if heat_data:
        HeatMap(heat_data, min_opacity=0.4, radius=20, blur=10, max_zoom=1).add_to(myanmar_map)

    # 🔹 **Add Earthquake Markers**
    for _, row in filtered_eq.iterrows():
        weighted_population = row["Weighted_Affected_Pop"]

        folium.CircleMarker(
            location=[row["latitude"], row["longitude"]],
            radius=max(5, np.log1p(weighted_population) / 2),
            color="red",
            fill=True,
            fill_color="red",
            fill_opacity=min(0.9, weighted_population / 1e6) if weighted_population > 0 else 0.3,
            popup=(
                f"<b>Date:</b> {row['time'].date()}<br>"
                f"<b>Magnitude:</b> {row['mag']}<br>"
                f"<b>Estimated Affected Population:</b> {int(weighted_population):,}"
            ),
        ).add_to(myanmar_map)

    display(myanmar_map)


# 🔹 **Interactive Widget for Filtering (Only Date Selection)**
default_start_date = datetime.today()
default_end_date = datetime.today()

date_picker = widgets.DatePicker(
    description="Start Date",
    value=default_start_date,  # Default to 2025-01-01
    layout={"width": "50%"}
)

end_date_picker = widgets.DatePicker(
    description="End Date",
    value=default_end_date,  # Default to today's date
    layout={"width": "50%"}
)

widgets.interactive(
    plot_earthquake_map,
    start_date=date_picker,
    end_date=end_date_picker
)


interactive(children=(DatePicker(value=datetime.datetime(2025, 3, 31, 2, 31, 17, 439049), description='Start D…